In [1]:
COSMOS_ENDPOINT = 'https://azcdb-dam.documents.azure.com:443/'
COSMOS_KEY = '0EgYze9fkgOGddU8AnolzDrgC7RuGy8LdK4v8By8h6Pk3y39axqaQ2wI0LmWw79lXnVh67En9AhEACDbi8iLzg=='

DATABASE_NAME = 'test_db'
CONTAINER_NAME = 'retail'

In [50]:
from azure.cosmos import CosmosClient, exceptions, PartitionKey

In [51]:
client = CosmosClient(COSMOS_ENDPOINT, COSMOS_KEY)

In [4]:
database = client.create_database_if_not_exists(id=DATABASE_NAME)

In [6]:
container = database.create_container_if_not_exists(
    id=CONTAINER_NAME,
    partition_key=PartitionKey(path='/categoria'),
    offer_throughput=400
    )

In [28]:
documento = {
    "id":"8",
    "nombre":"Producto H",
    "categoria":"Deportes",
    "precio":85.00

}
type(documento)

dict

In [29]:
try:
    container.create_item(body=documento)
    print("Documento insertado exitosamente.")
except exceptions.CosmosResourceExistsError:
    print("El documento ya existe.")

Documento insertado exitosamente.


In [32]:
# Store Procedures

list_sp = container.scripts.list_stored_procedures()

for i in list_sp:
    print (i)

{'id': 'incrementarPrecio', 'body': 'function incrementarPrecio(categoria, incremento) {\n    var context = getContext();\n    var collection = context.getCollection();\n    var response = context.getResponse();\n\n    var query = {\n        query: "SELECT * FROM c WHERE c.categoria = @categoria",\n        parameters: [{ name: "@categoria", value: categoria }]\n    };\n\n    var accept = collection.queryDocuments(\n        collection.getSelfLink(),\n        query,\n        function (err, documents, responseOptions) {\n            if (err) throw new Error("Error al consultar documentos: " + err.message);\n\n            if (documents.length > 0) {\n                for (var i = 0; i < documents.length; i++) {\n                    var doc = documents[i];\n                    doc.precio += parseInt(incremento, 10);\n\n                    var acceptUpdate = collection.replaceDocument(doc._self, doc, function (err, docReplaced) {\n                        if (err) throw new Error("Error al act

In [33]:
sp = container.scripts.get_stored_procedure('incrementarPrecio')

sp

{'id': 'incrementarPrecio',
 'body': 'function incrementarPrecio(categoria, incremento) {\n    var context = getContext();\n    var collection = context.getCollection();\n    var response = context.getResponse();\n\n    var query = {\n        query: "SELECT * FROM c WHERE c.categoria = @categoria",\n        parameters: [{ name: "@categoria", value: categoria }]\n    };\n\n    var accept = collection.queryDocuments(\n        collection.getSelfLink(),\n        query,\n        function (err, documents, responseOptions) {\n            if (err) throw new Error("Error al consultar documentos: " + err.message);\n\n            if (documents.length > 0) {\n                for (var i = 0; i < documents.length; i++) {\n                    var doc = documents[i];\n                    doc.precio += parseInt(incremento, 10);\n\n                    var acceptUpdate = collection.replaceDocument(doc._self, doc, function (err, docReplaced) {\n                        if (err) throw new Error("Error al ac

In [41]:
sp_exec = container.scripts.execute_stored_procedure(sp,partition_key='Deportes',params=['Deportes',10])

sp_exec

In [42]:
list_tr = container.scripts.list_triggers()

for i in list_tr:
    print (i)

{'id': 'validarDatos', 'body': 'function validarDatosAntesInsercion() {\r\n    var context = getContext();\r\n    var request = context.getRequest();\r\n    var documento = request.getBody();\r\n\r\n    if (documento.precio < 0) {\r\n        throw new Error("El precio del producto no puede ser negativo.");\r\n    }\r\n}', 'triggerOperation': 'Create', 'triggerType': 'Pre', '_rid': '5QE4APi8h+wBAAAAAAAAcA==', '_self': 'dbs/5QE4AA==/colls/5QE4APi8h+w=/triggers/5QE4APi8h+wBAAAAAAAAcA==/', '_etag': '"8801d5bb-0000-0200-0000-6716e0cd0000"', '_ts': 1729552589}


In [44]:
tr = container.scripts.get_trigger('validarDatos')

tr

{'id': 'validarDatos',
 'body': 'function validarDatosAntesInsercion() {\r\n    var context = getContext();\r\n    var request = context.getRequest();\r\n    var documento = request.getBody();\r\n\r\n    if (documento.precio < 0) {\r\n        throw new Error("El precio del producto no puede ser negativo.");\r\n    }\r\n}',
 'triggerOperation': 'Create',
 'triggerType': 'Pre',
 '_rid': '5QE4APi8h+wBAAAAAAAAcA==',
 '_self': 'dbs/5QE4AA==/colls/5QE4APi8h+w=/triggers/5QE4APi8h+wBAAAAAAAAcA==/',
 '_etag': '"8801d5bb-0000-0200-0000-6716e0cd0000"',
 '_ts': 1729552589}

In [ ]:
documento = {
    "id":"9",
    "nombre":"Producto I",
    "categoria":"Deportes",
    "precio":-5.00

}

In [49]:
try:
    container.create_item(documento,pre_trigger_include='validarDatos')
    print("Documento insertado exitosamente.")
except Exception as e:
    print("Error al crear el item: ",e)

Error al crear el item:  (Conflict) Entity with the specified id already exists in the system., 
RequestStartTime: 2024-10-21T23:47:58.0638868Z, RequestEndTime: 2024-10-21T23:47:58.0683503Z,  Number of regions attempted:1
{"systemHistory":[{"dateUtc":"2024-10-21T23:47:07.2610111Z","cpu":0.824,"memory":617521324.000,"threadInfo":{"isThreadStarving":"False","threadWaitIntervalInMs":0.0414,"availableThreads":32765,"minThreads":64,"maxThreads":32767},"numberOfOpenTcpConnection":5923},{"dateUtc":"2024-10-21T23:47:17.2710511Z","cpu":0.163,"memory":617511156.000,"threadInfo":{"isThreadStarving":"False","threadWaitIntervalInMs":0.0399,"availableThreads":32765,"minThreads":64,"maxThreads":32767},"numberOfOpenTcpConnection":5923},{"dateUtc":"2024-10-21T23:47:27.2810946Z","cpu":0.244,"memory":617521696.000,"threadInfo":{"isThreadStarving":"False","threadWaitIntervalInMs":0.0289,"availableThreads":32765,"minThreads":64,"maxThreads":32767},"numberOfOpenTcpConnection":5924},{"dateUtc":"2024-10-21T23